In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret found; cloning this PRIVATE repo will fail.\n"
                  "Add a GitHub token (repo scope) via the key icon (Secrets) as 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok
                else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_root)], check=True)
        subprocess.run(["git", "-C", str(_root), "remote", "set-url", "origin",
                        f"https://github.com/{_slug}.git"])  # keep the token out of the saved remote
    os.chdir(_root / "07-application-agent-framework/long-running-durable/lra-core/lra-core/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 03 · Retries, crashes and the reaper

What happens when the model 503s, when the worker dies at the worst moment, and when two workers collide.

In [ ]:
import sys; sys.path[:0] = ["..", "../.."]      # repo root, from notebooks/ or notebooks/solutions/
from core import Engine, Queue, Store, Crash, drain
from workflow import STEPS, CALLS
clock = [0.0]; now = lambda: clock[0]              # a clock we control: days pass in one line
def fresh(steps=STEPS):
    CALLS.clear(); return Engine(Store(), Queue(), dict(steps), clock=now, lease_ttl=60)


## 1. A flaky step

Model APIs return 503s. The engine treats any `Exception` as retryable: it bumps the attempt (so the old task is stale by construction) and enqueues a delayed retry.

In [ ]:
# TODO: fill in: attempts
calls = []
def flaky(ctx):
    calls.append(1)
    if len(calls) < 3:
        raise ConnectionError("503 model overloaded")
    return ("done", "ok")

engine = fresh({"flaky": flaky})
engine.start("r", "flaky", {})
print(drain(engine, engine.queue, now), engine.store.get("r")["attempts"], "next due at", engine.queue.tasks[0][0])
clock[0] += 2;  print(drain(engine, engine.queue, now), engine.store.get("r")["attempts"])
clock[0] += 4;  print(drain(engine, engine.queue, now))
assert engine.store.get("r")["attempts"] == {"flaky": ____} and engine.store.get("r")["status"] == "SUCCEEDED"

## 2. Giving up

After `max_attempts` the run fails with a readable error. What does the run look like?

In [ ]:
# TODO: fill in: status
engine = fresh({"bad": lambda ctx: 1 / 0})
engine.start("r", "bad", {})
for _ in range(3):
    drain(engine, engine.queue, now); clock[0] += 10
run = engine.store.get("r")
print(run["status"], "|", run["error"]); print(run["history"])
assert run["status"] == ____ and len(run["history"]) == 3

## 3. The crash the queue cannot see

The worker dies **after** the checkpoint and **before** enqueueing the next step. Cloud Tasks redelivers the old task (stale). Nobody enqueues the new one. Who repairs this, and when?

In [ ]:
# TODO: fill in: seconds
engine = fresh()
engine.crash_before_enqueue = True
engine.start("run-1", "draft", {"topic": "x"})
print(drain(engine, engine.queue, now))
run = engine.store.get("run-1"); print("step:", run["step"], "| lease until:", run["lease"])
clock[0] += 30; print("redelivery:", drain(engine, engine.queue, now))
print("reap now:", engine.reap())
clock[0] += ____
print("reap later:", engine.reap())
print(drain(engine, engine.queue, now))
assert engine.store.get("run-1")["status"] == "WAITING"

## 4. Two workers, one run

A second replica receives a duplicate delivery while the first is mid-step. The lease makes it back off (HTTP 503 → Cloud Tasks retries later).

In [ ]:
# TODO: fill in: after
engine = fresh()
engine.start("run-1", "draft", {"topic": "x"})
run = engine.store.get("run-1"); run["lease"] = now() + 60; engine.store.save(run)   # replica A holds the lease
print(engine.execute("run-1", "draft", 1))
clock[0] += 61                                                                       # A died; lease expired
assert engine.execute("run-1", "draft", 1) == ____

## 5. Your turn: a step that must never be retried

A business-rule failure (customer on a sanctions list) is not an outage. Make `check` finish the run as `FAILED`-by-design **without** burning retries. (Hint: it is an outcome, not an exception.)

In [ ]:
# TODO: fill in: kind
def check(ctx):
    if ctx.input["customer"] in {"acme-blocked"}:
        return (____, {"ok": False, "reason": "sanctions"})
    return ("next", "ship")
engine = fresh({"check": check, "ship": lambda ctx: ("done", {"ok": True})})
engine.start("o1", "check", {"customer": "acme-blocked"})
print(drain(engine, engine.queue, now))
run = engine.store.get("o1")
assert run["status"] == "SUCCEEDED" and run["result"]["ok"] is False and run["attempts"] == {"check": 1}